# 📄 Case Study: Semantic Document Search

## What We're Building

A search engine that finds documents by **meaning**, not just keywords.

```
Keyword search: "machine learning model"  →  finds docs with those exact words
Semantic search: "ML model"               →  also finds docs about "neural networks",
                                              "AI training", "deep learning" etc.
```

Real use cases: internal wikis, research paper search, legal document discovery.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

Ready!


In [2]:
# Simulated document library
# Each doc has a title + body

library = [
    {"id": 1, "title": "Intro to Neural Networks",     "body": "Neural networks are ML models made of layers of connected neurons that learn patterns from data."},
    {"id": 2, "title": "Python for Beginners",         "body": "Python is a high-level programming language with clean syntax, great for data science and automation."},
    {"id": 3, "title": "How Transformers Work",        "body": "Transformers use self-attention to process sequences in parallel, revolutionising NLP tasks."},
    {"id": 4, "title": "Database Design Basics",       "body": "Relational databases store data in tables with rows and columns, using SQL for queries."},
    {"id": 5, "title": "Training Deep Learning Models","body": "Training involves forward pass, loss calculation, backpropagation, and gradient-based weight updates."},
    {"id": 6, "title": "REST API Design Guide",        "body": "REST APIs use HTTP methods like GET and POST to expose resources as URLs."},
    {"id": 7, "title": "Overfitting and Regularization","body": "Overfitting is when a model memorizes training data. Dropout and L2 are common regularization methods."},
    {"id": 8, "title": "Vector Databases Explained",   "body": "Vector databases like Pinecone and Chroma store embeddings and support fast similarity search."},
]

# Embed title + body together for richer search
combined_texts = [f"{d['title']}. {d['body']}" for d in library]
embeddings = model.encode(combined_texts)

print(f"Indexed {len(library)} documents")

Indexed 8 documents


In [3]:
# Semantic search function

def search(query, top_k=3):
    q_emb  = model.encode(query)
    scores = cosine_similarity([q_emb], embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    print(f"Query: '{query}'")
    print("-" * 55)
    for rank, idx in enumerate(top_idx, 1):
        doc = library[idx]
        print(f"  {rank}. [{scores[idx]:.3f}] {doc['title']}")
        print(f"       {doc['body'][:70]}...")
    print()

search("how do neural networks learn?")
search("prevent model from memorizing training data")
search("build a web service endpoint")

Query: 'how do neural networks learn?'
-------------------------------------------------------
  1. [0.653] Intro to Neural Networks
       Neural networks are ML models made of layers of connected neurons that...
  2. [0.574] Training Deep Learning Models
       Training involves forward pass, loss calculation, backpropagation, and...
  3. [0.314] How Transformers Work
       Transformers use self-attention to process sequences in parallel, revo...

Query: 'prevent model from memorizing training data'
-------------------------------------------------------
  1. [0.317] Overfitting and Regularization
       Overfitting is when a model memorizes training data. Dropout and L2 ar...
  2. [0.301] Training Deep Learning Models
       Training involves forward pass, loss calculation, backpropagation, and...
  3. [0.245] Intro to Neural Networks
       Neural networks are ML models made of layers of connected neurons that...

Query: 'build a web service endpoint'
-----------------------------

In [4]:
# Keyword vs Semantic — side by side comparison

def keyword_search(query, top_k=3):
    query_words = set(query.lower().split())
    scored = []
    for i, d in enumerate(library):
        doc_words = set((d['title'] + ' ' + d['body']).lower().split())
        overlap = len(query_words & doc_words)
        scored.append((overlap, i))
    scored.sort(reverse=True)
    return [idx for _, idx in scored[:top_k]]

test_query = "AI training prevent overfitting"

kw_results  = keyword_search(test_query)
q_emb       = model.encode(test_query)
scores      = cosine_similarity([q_emb], embeddings)[0]
sem_results = list(np.argsort(scores)[::-1][:3])

print(f"Query: '{test_query}'\n")
print(f"{'Keyword Search':<40} {'Semantic Search'}")
print("-" * 80)
for k, s in zip(kw_results, sem_results):
    print(f"{library[k]['title']:<40} {library[s]['title']}")

Query: 'AI training prevent overfitting'

Keyword Search                           Semantic Search
--------------------------------------------------------------------------------
Overfitting and Regularization           Overfitting and Regularization
Training Deep Learning Models            Training Deep Learning Models
Vector Databases Explained               Intro to Neural Networks


## Key Takeaway

Semantic search finds documents based on **meaning** — even when the exact words don't match.

The only thing that changed from basic RAG:
- We embed `title + body` together → richer signal
- We show metadata (title) alongside results → more useful to users

Apply this pattern to any document library: PDFs, wikis, notes, research papers.